In [ ]:
%pip install pyautogen 

In [14]:
%pip install python-dotenv

In [15]:
%load_ext dotenv
%dotenv

In [ ]:
import os
from autogen import AssistantAgent, UserProxyAgent
from autogen import ConversableAgent
from autogen.coding import CodeBlock, LocalCommandLineCodeExecutor
from pathlib import Path


llm_config = { "config_list": [{
        "model": "gpt-4o",
        "api_key": os.getenv("API_KEY"),
        "base_url" : os.getenv("BASE_URL"),
        "api_version": "2024-08-01-preview"
    }] }

assistant = AssistantAgent("assistant", llm_config=config_list, 
                            code_execution_config=False,
                            max_consecutive_auto_reply=30,
                            human_input_mode="NEVER"
                            )

task_message = """
Deploy the following Sybase store procedure as Postgres to the local postgres database test_autogen with username postgres and password postgres.

Use only python. Must be a stored procedure and not a function.

Once the Postgres stored procedure has been created, generate a file test_TopCustomer.sql that contains a set of comprehensive tests for the stored procedure.
The unit tests should test all reasonable combination of inputs including possible corner cases.

Execute the unit tests in the test_TopCustomer.sql file and output the results to a file test_TopCustomer_results.txt

test_TopCustomer_results.txt should be in the following format:
<test name>,<test description>,<sql code for the test>,<test inputs>,<expected output for the test>,<actual observed output of the test>,<pass or fail result>

create procedure increase_price
as

    /* start the transaction */
    begin transaction
    /* first update prices > $60 */
    update titles
        set price = price * 1.05
        where price > $60

    /* next, prices between $30 and $60 */
    update titles 
        set price = price * 1.10    
    where price > $30 and price <= $60

    /* and finally prices <= $30 */
    update titles 
    set price = price * 1.20
    where price <= $30

    /* commit the transaction */ 
    commit transaction

return
"""

work_dir = Path("code")
work_dir.mkdir(exist_ok=True)

executor = LocalCommandLineCodeExecutor(work_dir=work_dir)


code_executor_agent = ConversableAgent(
    name="code_executor_agent",
    llm_config=False,
    code_execution_config={
        "executor": executor,
    },
    human_input_mode="NEVER",
    #is_termination_msg=lambda msg: "terminate" in msg["content"].lower(),
)

assistant.initiate_chat(code_executor_agent, message=task_message)

assistant (to code_executor_agent):


Deploy the following Sybase store procedure as Postgres to the local postgres database test_autogen with username postgres and password postgres.

Use only python. Must be a stored procedure and not a function.

Once the Postgres stored procedure has been created, generate a file test_TopCustomer.sql that contains a set of comprehensive tests for the stored procedure.
The unit tests should test all reasonable combination of inputs including possible corner cases.

Execute the unit tests in the test_TopCustomer.sql file and output the results to a file test_TopCustomer_results.txt

test_TopCustomer_results.txt should be in the following format:
<test name>,<test description>,<sql code for the test>,<test inputs>,<expected output for the test>,<actual observed output of the test>,<pass or fail result>

create procedure increase_price
as

    /* start the transaction */
    begin transaction
    /* first update prices > $60 */
    update titles
      